In [1]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import sys, os
from autograd import grad, hessian
import autograd.numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV,PredefinedSplit
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
from sklearn.metrics import mean_squared_error
import matplotlib.colors as mcolors
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor
import joblib


In [2]:
# Load the data 
data = np.load('../../data_splits_splot22f_1215.npz')
X_train = data['X_train']
y_train = data['y_train'][:, 1:] #grabbing only regression labelss
X_val = data['X_val']
y_val = data['y_val'][:, 1:]
X_test = data['X_test']
y_test = data['y_test'][:, 1:]

#Standard normalize the training data and use the mean and std to normalize the testing data
knn_scaler = StandardScaler().fit(X_train)
X_train = knn_scaler.transform(X_train)
X_test = knn_scaler.transform(X_test)
X_val = knn_scaler.transform(X_val)

print("Training set ", np.shape(X_train), np.shape(y_train))
print("Validation set ", np.shape(X_val), np.shape(y_val))
print("Testing set ", np.shape(X_test), np.shape(y_test))



Training set  (19404, 5) (19404, 3)
Validation set  (4158, 5) (4158, 3)
Testing set  (4158, 5) (4158, 3)


In [3]:
# Create a split indicator array
split_index = np.concatenate([
    np.full(len(X_train), -1),  # All train samples get -1
    np.zeros(len(X_val))         # All val samples get 0
])

# Combine train and validation sets
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val], axis=0)

print(np.shape(y_train_val))
# Create the predefined split
ps = PredefinedSplit(test_fold=split_index)

estimator_KNN = KNeighborsRegressor(algorithm='auto')

parameters_KNN = {
    'n_neighbors': [ 3, 5, 7, 9, 11, 15, 20, 25],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']}
grid_search_KNN = GridSearchCV(
    estimator= estimator_KNN,
    param_grid=parameters_KNN,
    scoring = 'neg_mean_absolute_error',
    cv = ps)


grid_search_KNN.fit(X_train_val, y_train_val)

# Performance on test dataset
best_model_knn = grid_search_KNN.best_estimator_

y_pred_train = best_model_knn.predict(X_train)
y_pred_val = best_model_knn.predict(X_val)
y_pred_test = best_model_knn.predict(X_test)


# Calculate absolute and relative errors for each mass component 
X_train_unnormalized = knn_scaler.inverse_transform(X_train)
X_val_unnormalized = knn_scaler.inverse_transform(X_val)
X_test_unnormalized = knn_scaler.inverse_transform(X_test)

mass1i_train, mass1i_val, mass1i_test  = (np.exp(X_train_unnormalized[:, 3]), np.exp(X_val_unnormalized[:, 3]), np.exp(X_test_unnormalized[:, 3]))
mass2i_train, mass2i_val, mass2i_test  = (np.exp(X_train_unnormalized[:, 4]), np.exp(X_val_unnormalized[:, 4]), np.exp(X_test_unnormalized[:, 4]))

initial_total_masses_train = mass1i_train + mass2i_train # in Msun 
initial_total_masses_val = mass1i_val + mass2i_val # in Msun 
initial_total_masses_test = mass1i_test + mass2i_test # in Msun 

pred_mass1_train, pred_mass1_val, pred_mass1_test = (y_pred_train[:,0] * initial_total_masses_train, y_pred_val[:,0] * initial_total_masses_val, y_pred_test[:,0] * initial_total_masses_test)
pred_mass2_train, pred_mass2_val, pred_mass2_test = (y_pred_train[:,1] * initial_total_masses_train, y_pred_val[:,1] * initial_total_masses_val, y_pred_test[:,1] * initial_total_masses_test)
pred_ejec_test  = y_pred_test[:,2] * initial_total_masses_test #we don't need this for train or val sets 

true_mass1_train, true_mass1_val, true_mass1_test = (y_train[:,0] * initial_total_masses_train, y_val[:,0] * initial_total_masses_val, y_test[:,0] * initial_total_masses_test)
true_mass2_train, true_mass2_val, true_mass2_test = (y_train[:,1] * initial_total_masses_train, y_val[:,1] * initial_total_masses_val, y_test[:,1] * initial_total_masses_test)
true_ejec_test    =  y_test[:,2] * initial_total_masses_test

# Save everything you'll need for plotting
np.savez('../results/knn_results.npz',
         y_pred_train=[pred_mass1_train, pred_mass2_train],
         y_pred_val=[pred_mass1_val, pred_mass2_val],
         y_pred_test=[pred_mass1_test, pred_mass2_test], 
         y_true_train=[true_mass1_train, true_mass2_train],
         y_true_val=[true_mass1_val, true_mass2_val],
         y_true_test=[true_mass1_test, true_mass2_test])


#--Error metric 1: Median Absolute Errors for the respective masses 
median_abs_error_m1_test = np.median(np.abs(pred_mass1_test - true_mass1_test))
median_abs_error_m2_test = np.median(np.abs(pred_mass2_test - true_mass2_test))
median_abs_error_ejec_test = np.median(np.abs(pred_ejec_test - true_ejec_test))

mean_abs_error_m1_test = np.mean(np.abs(pred_mass1_test - true_mass1_test))
mean_abs_error_m2_test = np.mean(np.abs(pred_mass2_test - true_mass2_test))

#--Error metric 2: Relative errors for cases where at least one star survives
median_rel_error_m1_test = np.median(np.abs(pred_mass1_test[true_mass1_test != 0.] - true_mass1_test[true_mass1_test != 0. ])/ true_mass1_test[true_mass1_test != 0.])
median_rel_error_m2_test = np.median(np.abs(pred_mass2_test[true_mass2_test != 0.] - true_mass2_test[true_mass2_test != 0. ])/ true_mass2_test[true_mass2_test != 0.])
median_rel_error_m_ejec_test = np.median(np.abs(pred_ejec_test[true_ejec_test != 0.] - true_ejec_test[true_ejec_test != 0. ])/ true_ejec_test[true_ejec_test != 0.])

mean_rel_error_m1_test = np.mean(np.abs(pred_mass1_test[true_mass1_test != 0.] - true_mass1_test[true_mass1_test != 0. ])/ true_mass1_test[true_mass1_test != 0.])
mean_rel_error_m2_test = np.mean(np.abs(pred_mass2_test[true_mass2_test != 0.] - true_mass2_test[true_mass2_test != 0. ])/ true_mass2_test[true_mass2_test != 0.])


print(f"Median Absolute Errors M1 [Msun]: {median_abs_error_m1_test:.4f}")
print(f"Median bsolute Errors M2 [Msun]: {median_abs_error_m2_test:.8f}")
print(f"Mean Absolute Errors M1 [Msun]: {mean_abs_error_m1_test:.4f}")
print(f"Mean Absolute Errors M2 [Msun]: {mean_abs_error_m2_test:.4f}")

print(f"Median Relative Errors M1,f  : {median_rel_error_m1_test:.4f}")
print(f"Median Relative Errors M2,f: {median_rel_error_m2_test:.8f}")
print(f"Mean Relative Errors M1,f : {mean_rel_error_m1_test:.4f}")
print(f"Mean Relative Errors M2,f: {mean_rel_error_m2_test:.4f}")

# Save the best-performing model 
# Also save best parameters for reference
save_obj = {
    'model': grid_search_KNN.best_estimator_, 
    'scaler': knn_scaler,
    'best_params': grid_search_KNN.best_params_,
    'best_score': grid_search_KNN.best_score_,
    'cv_results': grid_search_KNN.cv_results_,
    'test_median_abs_errors_m1': median_abs_error_m1_test,
    'test_median_abs_errors_m2': median_abs_error_m2_test, 
    'test_median_rel_errors_m1': median_rel_error_m1_test, 
    'test_median_rel_errors_m2' :median_rel_error_m2_test}

joblib.dump(save_obj, "../best_models/knn_best_model.pkl")


(23562, 3)
Median Absolute Errors M1 [Msun]: 0.0210
Median bsolute Errors M2 [Msun]: 0.00001012
Mean Absolute Errors M1 [Msun]: 0.5692
Mean Absolute Errors M2 [Msun]: 0.2794
Median Relative Errors M1,f  : 0.0037
Median Relative Errors M2,f: 0.01123096
Mean Relative Errors M1,f : 0.0725
Mean Relative Errors M2,f: 0.1504


['../best_models/knn_best_model.pkl']

In [2]:

model_name = '../best_models/knn_best_model.pkl'
checkpoint = joblib.load(model_name)
params = checkpoint["best_params"]
score = checkpoint["best_score"]
print(params)

{'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}


In [5]:
def knn_regression_plotter(X_train, y_train, X_test, y_test, best_model_knn, knn_scaler,
                       feature_idx=(0, 1), fixed_values={}, labels=[] ):
    """
    Plots regression outputs for a PyTorch model using a 2D slice of a higher-dimensional space.
    Produces one panel per output dimension (color gradient).
    
    Parameters:
    - best_model_knn: Trained PyTorch model.
    - X_train, y_train, X_test, y_test: datasets
    - knn_scaler: normalization stats.
    - feature_idx: Tuple (i, j) specifying which two features to plot.
    - fixed_values: {feature_index: value} for fixing other dimensions.
    - labels: list of strings for axis and titles. Expected: [x_label, y_label, ..., etc].
    """
    # Step 1: Normalize fixed values

    fixed_values_norm = {k: (v - knn_scaler.mean_[k]) / knn_scaler.scale_[k] for k, v in fixed_values.items()}

    train_mask = np.all(np.array([np.isclose(X_train[:, k], v, rtol=0.09) for k, v in fixed_values_norm.items()]), axis=0)
    test_mask = np.all(np.array([np.isclose(X_test[:, k], v, rtol=0.09) for k, v in fixed_values_norm.items()]), axis=0)
   
    if (train_mask.sum() == 0 and test_mask.sum() == 0):
        print("No data points!")
        return "No data points!"

    X_train_filtered, y_train_filtered = X_train[train_mask], y_train[train_mask]
    X_test_filtered, y_test_filtered = X_test[test_mask], y_test[test_mask]

    #Convert masses into not logged 
    X_train_filtered_unnorm = knn_scaler.inverse_transform(X_train_filtered)
    X_test_filtered_unnorm = knn_scaler.inverse_transform(X_test_filtered)
    X_train_mass1 = np.exp(X_train_filtered_unnorm[:,3])
    X_train_mass2 = np.exp(X_train_filtered_unnorm[:,4])

    X_test_mass1 = np.exp(X_test_filtered_unnorm[:,3])
    X_test_mass2 = np.exp(X_test_filtered_unnorm[:,4])

    y_train_filtered_unnorm = np.array(y_train_filtered) 
    y_test_filtered_unnorm = np.array(y_test_filtered) 

    # Correct Units 
    y_train_filtered_unnorm[:,0] = y_train_filtered_unnorm[:,0] * (X_train_mass1 + X_train_mass2)
    y_train_filtered_unnorm[:,1] = y_train_filtered_unnorm[:,1] * (X_train_mass1 + X_train_mass2)

    y_test_filtered_unnorm[:,0] = y_test_filtered_unnorm[:,0] * (X_test_mass1 + X_test_mass2)
    y_test_filtered_unnorm[:,1] = y_test_filtered_unnorm[:,1] * (X_test_mass1 + X_test_mass2)
    
    x_min, x_max = X_train_filtered[:, feature_idx[0]].min() - 0.1, X_train_filtered[:, feature_idx[0]].max() + 0.1
    y_min, y_max = X_train_filtered[:, feature_idx[1]].min() - 0.1, X_train_filtered[:, feature_idx[1]].max() + 0.1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                         np.linspace(y_min, y_max, 500))

    # Step 3: Construct full-dimensional input space for predictions
    X_grid = np.zeros((xx.ravel().shape[0], X_train.shape[1]))
    X_grid[:, feature_idx[0]] = xx.ravel()
    X_grid[:, feature_idx[1]] = yy.ravel()
    
    for k, v in fixed_values.items():
        X_grid[:, k] = v

    #transform the data
    X_grid_scaled = knn_scaler.transform(X_grid)

    #Predict labels for meshgrid
    preds = best_model_knn.predict(X_grid_scaled)
    preds = preds.reshape(xx.shape)

    # Changing the training and testing data to be M1f and M2f 
    y_train_filtered_corrected = np.empty((len(y_train_filtered_unnorm[:,0]), 2))
    y_test_filtered_corrected = np.empty((len(y_test_filtered_unnorm[:,0]), 2))
    
    y_train_filtered_corrected[:,0] = y_train_filtered_unnorm[:, 0]
    y_train_filtered_corrected[:,1] = y_train_filtered_unnorm[:, 1] 

    y_test_filtered_corrected[:,0] = y_test_filtered_unnorm[:, 0]
    y_test_filtered_corrected[:,1] = y_test_filtered_unnorm[:, 1] 

    M1_f = preds[:,0] * (np.exp(fixed_values[3]) + np.exp(fixed_values[4]))
    M2_f = preds[:,1] * (np.exp(fixed_values[3]) + np.exp(fixed_values[4]))

    # Reshape into grid for each output dimension
    M1_f = M1_f.reshape(xx.shape)
    M2_f = M2_f.reshape(xx.shape)

    # Step 5: Plot two panels
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    vmin = min(y_train_filtered_corrected[:,0].min(), y_train_filtered_corrected[:,1].min(),
           y_test_filtered_corrected[:,0].min(), y_test_filtered_corrected[:,1].min(),
           M1_f.min(), M2_f.min())
    vmax = max(y_train_filtered_corrected[:,0].max(), y_train_filtered_corrected[:,1].max(),
           y_test_filtered_corrected[:,0].max(), y_test_filtered_corrected[:,1].max(),
           M1_f.max(), M2_f.max())
    
    cmap = plt.cm.coolwarm
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for i, (Z, ax, title) in enumerate(zip([M1_f, M2_f], axes, ['Star 1 Final Mass', 'Star 2 Final Mass'])):
        im = ax.contourf(xx, yy, Z, levels = 100, cmap=cmap, norm = norm)
        scatter1 = ax.scatter(X_train_filtered[:, feature_idx[0]],
                              X_train_filtered[:, feature_idx[1]],
                              c=y_train_filtered_corrected[:, i], cmap=cmap, norm = norm, edgecolor="k", marker="o", label="Train")
        scatter2 = ax.scatter(X_test_filtered[:, feature_idx[0]],
                              X_test_filtered[:, feature_idx[1]],
                              c=y_test_filtered_corrected[:, i], cmap=cmap, norm = norm, marker="^", label="Test")

        ax.set_xlabel(fr"{labels[0]}", fontsize=14)
        ax.set_ylabel(fr"{labels[1]}", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.set_title(title, fontsize=15)
    
   
    # After plotting the contours and scatters
    plt.tight_layout(rect=[0,0,0.9,1])  # leave 10% space on the right for the colorbar

    # Create the colorbar on a dedicated axis outside the panels
    cbar_ax = fig.add_axes([0.9999, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                    cax=cbar_ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Final Mass', fontsize=14)

    fig.suptitle(fr"$\mathrm{{M_1}} = {labels[2]}\ M_\odot,\ \mathrm{{M_2}} = {labels[3]}\ M_\odot,\ \mathrm{{Time}} = {round(10**(float(labels[4])),3)}\ \mathrm{{Gyr}}$", fontsize = 17)

    plt.tight_layout()
    return fig




In [92]:
# unique_rows = unique_rows.T
import matplotlib.colors as mcolors
unique_rows = np.array([[1.0,1.0]])
for unique in unique_rows:
    Mass1 = np.log(unique[0])
    Mass2 = np.log(unique[1])
    # labels = ['log10(b[RSUN])', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    labels = ['log10(rp/(R1+R2))', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    fig = knn_regression_plotter(X_train, y_train, X_test ,y_test, best_model_knn, knn_scaler,feature_idx=(0, 1), fixed_values={3: Mass1, 4: Mass2}, labels = labels)
    plt.show()

ValueError: cannot reshape array of size 750000 into shape (500,500)